# Day 2: RAG, step 4: grounded answers with Gemini

## Step 1: Setup

In [1]:
import json
import logging
import os
import time

import numpy as np
import psycopg
from dotenv import load_dotenv
from google import genai
from google.genai import errors, types
from pgvector.psycopg import register_vector
from pydantic import BaseModel, Field

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
for noisy in ("httpx", "google_genai"):
    logging.getLogger(noisy).setLevel(logging.WARNING)
logger = logging.getLogger("rag")

load_dotenv("../.env")

EMBED_MODEL = "gemini-embedding-001"
CHAT_MODELS = ["gemini-3.5-flash-lite", "gemini-3.8-flash"]
TOP_K = 3
MIN_SCORE = 0.6
TEMPERATURE = 0.1
MAX_QUESTION_CHARS = 500

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
conn = psycopg.connect(
    f"postgresql://{os.environ['POSTGRES_USER']}:{os.environ['POSTGRES_PASSWORD']}"
    f"@{os.environ['POSTGRES_HOST']}:{os.environ['POSTGRES_PORT']}/{os.environ['POSTGRES_DB']}",
    autocommit=True,
)
register_vector(conn)
count = conn.execute("SELECT count(*) FROM weather_rag.chunks").fetchone()[0]
print(f"Connected. {count} chunks in the database.")

Connected. 219 chunks in the database.


## Step 2: Retrieve: question → top 3 chunks

In [2]:
def embed_question(question):
    result = client.models.embed_content(
        model=EMBED_MODEL,
        contents=[question],
        config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY", output_dimensionality=768),
    )
    vector = np.array(result.embeddings[0].values, dtype="float32")
    return vector / np.linalg.norm(vector)

def retrieve(question, top_k=TOP_K):
    rows = conn.execute("""
        SELECT c.chunk_id, d.title, d.source_url, c.page_start, c.page_end,
               c.section_title, c.content, 1 - (c.embedding <=> %(q)s) AS score
        FROM weather_rag.chunks c
        JOIN weather_rag.documents d USING (doc_id)
        ORDER BY c.embedding <=> %(q)s
        LIMIT %(k)s
    """, {"q": embed_question(question), "k": top_k}).fetchall()
    keys = ["chunk_id", "doc_title", "source_url", "page_start", "page_end", "section_title", "content", "score"]
    return [dict(zip(keys, row)) for row in rows]

for hit in retrieve("At what temperature is a heat wave declared for coastal stations?"):
    print(f"{hit['score']:.3f}  {hit['chunk_id']:18} {hit['section_title'][:70]}")

0.818  heat_faq-004       Q. What is a criterion for describing Heat Wave for coastal stations?
0.763  forecast_sop-187   7.3.1. Criterion for declaring heat wave
0.760  heat_faq-003       Q. What is criterion for declaring heat wave in India?


## Step 3: The prompt: rules + context + question

In [3]:
SYSTEM_RULES = """You are a weather information assistant for operations staff in India.
Answer using ONLY the information in the CONTEXT. Never use your own knowledge.

Rules:
1. If the CONTEXT does not contain the answer, set "found" to false and say briefly that the IMD documents do not cover it.
2. If it does, set "found" to true and answer in 1-4 short, plain sentences. Keep numbers and units exactly as written.
3. Put the chunk id of every chunk you used in "sources", e.g. ["heat_faq-004"]. Only use ids that appear in the CONTEXT.
4. Do not add advice, warnings or facts that are not in the CONTEXT."""

def build_context(hits):
    blocks = []
    for h in hits:
        pages = f"p.{h['page_start']}" if h["page_start"] == h["page_end"] else f"pp.{h['page_start']}-{h['page_end']}"
        blocks.append(f"[{h['chunk_id']}] ({h['doc_title']}, {pages})\n{h['content']}")
    return "\n\n---\n\n".join(blocks)

def build_prompt(question, hits):
    return f"CONTEXT:\n{build_context(hits)}\n\nQUESTION: {question}"

example_hits = retrieve("At what temperature is a heat wave declared for coastal stations?")
print("SYSTEM RULES:\n" + SYSTEM_RULES)
print("\n" + "=" * 70 + "\nPROMPT:\n" + build_prompt("At what temperature is a heat wave declared for coastal stations?", example_hits)[:1500] + "\n...")

SYSTEM RULES:
You are a weather information assistant for operations staff in India.
Answer using ONLY the information in the CONTEXT. Never use your own knowledge.

Rules:
1. If the CONTEXT does not contain the answer, set "found" to false and say briefly that the IMD documents do not cover it.
2. If it does, set "found" to true and answer in 1-4 short, plain sentences. Keep numbers and units exactly as written.
3. Put the chunk id of every chunk you used in "sources", e.g. ["heat_faq-004"]. Only use ids that appear in the CONTEXT.
4. Do not add advice, warnings or facts that are not in the CONTEXT.

PROMPT:
CONTEXT:
[heat_faq-004] (IMD: FAQ on Heat Wave, p.2)
Q. What is a criterion for describing Heat Wave for coastal stations?
When maximum temperature departure is 4.5°C or more from normal, Heat Wave may
be described provided actual maximum temperature is 37°C or more.

---

[forecast_sop-187] (IMD: SOP for Weather Forecasting and Warning, p.175)
7.3.1. Criterion for declaring heat 

## Step 4: The answer format (JSON)

In [4]:
class GroundedAnswer(BaseModel):
    found: bool = Field(description="True only if the CONTEXT contains the answer")
    answer: str = Field(description="The answer, or a short note that the documents do not cover it")
    sources: list[str] = Field(description="Chunk ids from the CONTEXT that the answer is based on")

print(json.dumps(GroundedAnswer.model_json_schema(), indent=2)[:600])

{
  "properties": {
    "found": {
      "description": "True only if the CONTEXT contains the answer",
      "title": "Found",
      "type": "boolean"
    },
    "answer": {
      "description": "The answer, or a short note that the documents do not cover it",
      "title": "Answer",
      "type": "string"
    },
    "sources": {
      "description": "Chunk ids from the CONTEXT that the answer is based on",
      "items": {
        "type": "string"
      },
      "title": "Sources",
      "type": "array"
    }
  },
  "required": [
    "found",
    "answer",
    "sources"
  ],
  "title": "Gro


## Step 5: Call Gemini (with retry and a backup model)

In [5]:
def generate(prompt, retries=3):
    last_error = None
    for model in CHAT_MODELS:
        for attempt in range(1, retries + 1):
            try:
                response = client.models.generate_content(
                    model=model,
                    contents=prompt,
                    config=types.GenerateContentConfig(
                        system_instruction=SYSTEM_RULES,
                        temperature=TEMPERATURE,
                        response_mime_type="application/json",
                        response_schema=GroundedAnswer,
                        automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
                    ),
                )
                if response.parsed is None:
                    raise ValueError(f"reply was not valid JSON: {response.text[:200]!r}")
                return response.parsed, model
            except errors.APIError as error:
                last_error = error
                if error.code == 404:
                    logger.warning("%s not available, trying the next model", model)
                    break
                if error.code in (429, 500, 503) and attempt < retries:
                    wait = 5 * attempt
                    logger.warning("%s: error %s, retry %d/%d in %ds", model, error.code, attempt, retries, wait)
                    time.sleep(wait)
                    continue
                logger.warning("%s failed (%s), trying the next model", model, error.code)
                break
            except ValueError as error:
                last_error = error
                logger.warning("%s: %s (attempt %d/%d)", model, error, attempt, retries)
                continue
    logger.error("All chat models failed: %s", last_error)
    raise RuntimeError("The answer service is temporarily unavailable.") from last_error

## Step 6: `ask()`: the full pipeline

In [6]:
NOT_COVERED = "I don't have information about that in the IMD reference documents."

def ask(question):
    question = (question or "").strip()
    if not question:
        return {"status": "invalid_input", "answer": "Please type a question.", "sources": []}
    if len(question) > MAX_QUESTION_CHARS:
        return {"status": "invalid_input",
                "answer": f"The question is too long (max {MAX_QUESTION_CHARS} characters).", "sources": []}

    hits = retrieve(question)
    best = hits[0]["score"] if hits else 0.0
    if best < MIN_SCORE:
        logger.info("Best score %.3f < %.2f: not calling Gemini", best, MIN_SCORE)
        return {"status": "not_covered", "answer": NOT_COVERED, "sources": [], "best_score": round(best, 3)}

    reply, model = generate(build_prompt(question, hits))

    by_id = {h["chunk_id"]: h for h in hits}
    valid = [s for s in reply.sources if s in by_id]
    invented = [s for s in reply.sources if s not in by_id]
    if invented:
        logger.warning("Gemini cited unknown sources %s - removed", invented)
    if not reply.found or not valid:
        return {"status": "not_covered", "answer": reply.answer if not reply.found else NOT_COVERED,
                "sources": [], "best_score": round(best, 3), "model": model}

    return {
        "status": "answered",
        "answer": reply.answer,
        "sources": [{"chunk_id": s, "document": by_id[s]["doc_title"],
                     "pages": f"{by_id[s]['page_start']}-{by_id[s]['page_end']}",
                     "url": by_id[s]["source_url"], "score": round(by_id[s]["score"], 3)} for s in valid],
        "best_score": round(best, 3),
        "model": model,
    }

def show(question):
    result = ask(question)
    print(f"Q: {question!r}")
    print(f"   status: {result['status']}" + (f"   (best score {result['best_score']})" if "best_score" in result else ""))
    print(f"   answer: {result['answer']}")
    for s in result["sources"]:
        print(f"   source: [{s['chunk_id']}] {s['document']}, pages {s['pages']}")
    print()
    return result

_ = show("At what temperature is a heat wave declared for coastal stations?")

Q: 'At what temperature is a heat wave declared for coastal stations?'
   status: answered   (best score 0.818)
   answer: For coastal stations, a heat wave may be described when the maximum temperature departure is 4.5ãC or more from normal, provided the actual maximum temperature is 37ãC or more.
   source: [heat_faq-004] IMD: FAQ on Heat Wave, pages 2-2
   source: [forecast_sop-187] IMD: SOP for Weather Forecasting and Warning, pages 175-175



## Step 7: Test questions (the PRD Day 2 deliverable)

In [7]:
TESTS = [
    ("At what temperature is a heat wave declared for coastal stations?", "answered", "heat_faq"),
    ("What counts as very heavy rainfall?", "answered", "forecast_sop"),
    ("What is a cyclonic storm and what wind speed does it have?", "answered", "cyclone_terms"),
    ("How is the onset of monsoon over Kerala declared?", "answered", "monsoon_faq"),
    ("What should I do to protect myself during a heat wave?", "answered", "heat_faq"),
    ("Who won the cricket world cup?", "not_covered", None),
    ("What was the maximum temperature in Mumbai yesterday?", "not_covered", None),
    ("", "invalid_input", None),
]

report = []
for question, expected_status, expected_doc in TESTS:
    result = show(question)
    docs = {s["chunk_id"].rsplit("-", 1)[0] for s in result["sources"]}
    ok = result["status"] == expected_status and (expected_doc is None or expected_doc in docs)
    report.append({"question": question[:55] or "(empty)", "expected": expected_status,
                   "got": result["status"], "source docs": ", ".join(sorted(docs)) or "-",
                   "PASS": "✅" if ok else "❌"})

import pandas as pd
summary = pd.DataFrame(report)
print(f"Passed {(summary['PASS'] == '✅').sum()} of {len(summary)}")
summary

Q: 'At what temperature is a heat wave declared for coastal stations?'
   status: answered   (best score 0.818)
   answer: For coastal stations, a heat wave may be described when the maximum temperature departure is 4.5QC or more from normal, provided the actual maximum temperature is 37QC or more.
   source: [heat_faq-004] IMD: FAQ on Heat Wave, pages 2-2
   source: [forecast_sop-187] IMD: SOP for Weather Forecasting and Warning, pages 175-175



Q: 'What counts as very heavy rainfall?'
   status: answered   (best score 0.743)
   answer: Very heavy rainfall is defined as 24 hours cumulative rainfall at a station measuring 115.6-204.4 mm (12-20 cm, or 99.0-99.9 percentile).
   source: [forecast_sop-168] IMD: SOP for Weather Forecasting and Warning, pages 24-25
   source: [cyclone_terms-159] IMD RSMC New Delhi: Cyclone terminology, pages 7-8



Q: 'What is a cyclonic storm and what wind speed does it have?'
   status: answered   (best score 0.743)
   answer: A cyclonic storm is a cyclonic disturbance in which the maximum average surface wind speed is in the range of 34 to 47 knots (62 to 88 kmph).
   source: [cyclone_terms-143] IMD RSMC New Delhi: Cyclone terminology, pages 3-3



Q: 'How is the onset of monsoon over Kerala declared?'
   status: answered   (best score 0.789)
   answer: The onset of monsoon over Kerala is declared if after 10th May, 60% of the 14 designated stations report rainfall of 2.5 mm or more for two consecutive days, provided specific wind field, Outgoing Longwave Radiation, and atmospheric circulation criteria are also met.
   source: [monsoon_faq-099] IMD: FAQ on Monsoon, pages 12-13
   source: [monsoon_faq-100] IMD: FAQ on Monsoon, pages 12-13
   source: [monsoon_faq-101] IMD: FAQ on Monsoon, pages 12-13



Q: 'What should I do to protect myself during a heat wave?'
   status: answered   (best score 0.749)
   answer: Drink sufficient water frequently, wear lightweight light-colored loose cotton clothes, and avoid going out in the sun especially between 12.00 noon and 3.00 p.m. Use protective gear like goggles, an umbrella or hat, and shoes or chappals when going outside. Rehydrate with ORS, homemade drinks like lassi, torani, lemon water, or buttermilk, and avoid alcohol, tea, coffee, and carbonated soft drinks.
   source: [heat_faq-041] IMD: FAQ on Heat Wave, pages 13-14
   source: [heat_faq-033] IMD: FAQ on Heat Wave, pages 10-11



INFO rag: Best score 0.515 < 0.60: not calling Gemini


Q: 'Who won the cricket world cup?'
   status: not_covered   (best score 0.515)
   answer: I don't have information about that in the IMD reference documents.



Q: 'What was the maximum temperature in Mumbai yesterday?'
   status: not_covered   (best score 0.652)
   answer: The IMD documents do not cover it.

Q: ''
   status: invalid_input
   answer: Please type a question.



Passed 8 of 8


,question,expected,got,source docs,PASS
0,At what temperature is a heat wave declared fo...,answered,answered,"forecast_sop, heat_faq",✅
1,What counts as very heavy rainfall?,answered,answered,"cyclone_terms, forecast_sop",✅
2,What is a cyclonic storm and what wind speed d...,answered,answered,cyclone_terms,✅
3,How is the onset of monsoon over Kerala declared?,answered,answered,monsoon_faq,✅
4,What should I do to protect myself during a he...,answered,answered,heat_faq,✅
5,Who won the cricket world cup?,not_covered,not_covered,-,✅
6,What was the maximum temperature in Mumbai yes...,not_covered,not_covered,-,✅
7,(empty),invalid_input,invalid_input,-,✅
